In [5]:
import json
import numpy as np
import pandas as pd
import warnings
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)

warnings.filterwarnings("ignore")

In [6]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH  = r"D:\2026\MockProject_062026_NhomAI\data\dataset_5core_template.json"        
TARGET_COL = "Care_Level"            
CORES_KEY  = "5_cores"          
TEST_SIZE  = 0.2                
VAL_SIZE   = 0.15              
MODEL_SAVE_DIR = r"D:\2026\MockProject_062026_NhomAI\model"  

In [7]:
print("=" * 60)
print("  LOAD & PREPROCESS DATA")
print("=" * 60)

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# Tự động đọc các core và feature từ toàn bộ dữ liệu
CORE_FEATURES = {}

for record in raw_data:
    cores = record.get("5_cores", record.get("5 cores", {}))

    for core_name, core_data in cores.items():
        if core_name not in CORE_FEATURES:
            CORE_FEATURES[core_name] = []

        for feat in core_data.keys():
            if feat not in CORE_FEATURES[core_name]:
                CORE_FEATURES[core_name].append(feat)

X_raw = {core_name: [] for core_name in CORE_FEATURES.keys()}
y_raw = []

for record in raw_data:
    cores = record.get("5_cores", record.get("5 cores", {}))
    for core_name, feat_list in CORE_FEATURES.items():
        core_data = cores.get(core_name, {})
        row = []
        for feat in feat_list:
            val = core_data.get(feat, 0)
            if val is None:
                if feat in ["Chronic Disease", "Smoking Status", "Gender"]:
                    val = "Unknown"
                else:
                    val = 0.0
            row.append(val)
        X_raw[core_name].append(row)
    y_raw.append(record.get(TARGET_COL))

print(f"Total records loaded: {len(raw_data):,}")

# ---- Encode Target ----
y_raw = np.array(y_raw)
le_target = LabelEncoder()
y = le_target.fit_transform(y_raw)
num_classes = len(le_target.classes_)
print(f"Target classes encoded: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}")
print(f"Class distribution: {dict(zip(le_target.classes_, np.bincount(y)))}")

# ---- Train / Test / Val Split (Stratified) ----
indices = np.arange(len(y))

# 1. Tách tập test
train_val_idx, test_idx = train_test_split(
    indices, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# 2. Tách tập train và validation từ tập còn lại
val_ratio = VAL_SIZE / (1 - TEST_SIZE)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=val_ratio, random_state=RANDOM_STATE, stratify=y[train_val_idx]
)

y_train = y[train_idx]
y_val   = y[val_idx]
y_test  = y[test_idx]

  LOAD & PREPROCESS DATA
Total records loaded: 9,989
Target classes encoded: {np.float64(0.0): np.int64(0), np.float64(1.0): np.int64(1), np.float64(2.0): np.int64(2)}
Class distribution: {np.float64(0.0): np.int64(5320), np.float64(1.0): np.int64(3373), np.float64(2.0): np.int64(1296)}


In [8]:
print("\n" + "=" * 60)
print("  TRAINING & COMPARING COHESIVE PIPELINES")
print("=" * 60)


def get_model_instance(name):
    if name == "Random Forest":
        return RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
    elif name == "Gradient Boosting":
        return GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=5, random_state=RANDOM_STATE)
    elif name == "XGBoost":
        return xgb.XGBClassifier(n_estimators=150, learning_rate=0.1, max_depth=6, random_state=RANDOM_STATE, n_jobs=-1, eval_metric="mlogloss", verbosity=0)
    elif name == "LightGBM":
        return lgb.LGBMClassifier(n_estimators=150, learning_rate=0.1, max_depth=6, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
    return None


def build_meta_features(data, current_base_models):
    meta = []
    for core_name in core_input_dims:
        prob = current_base_models[core_name].predict_proba(data[core_name])
        meta.append(prob)
    return np.hstack(meta)


pipeline_candidates = ["Random Forest", "Gradient Boosting", "XGBoost", "LightGBM"]
meta_results = []

for name in pipeline_candidates:
    print(f"\n{'-'*60}")
    print(f"  Training Stacked Pipeline: {name}")
    print(f"{'-'*60}")
    
    # 1. Huấn luyện 5 base models con tương ứng
    curr_base_models = {}
    print("  Base Models Accuracy on Test Set:")
    for core_name in core_input_dims:
        model = get_model_instance(name)
        model.fit(X_train_cores[core_name], y_train)
        curr_base_models[core_name] = model
        
        pred = model.predict(X_test_cores[core_name])
        acc = accuracy_score(y_test, pred)
        print(f"    {core_name:<15}: {acc:.4f}")
        
    # 2. Tạo tập đặc trưng Meta cho Train, Val và Test
    X_train_meta = build_meta_features(X_train_cores, curr_base_models)
    X_val_meta   = build_meta_features(X_val_cores, curr_base_models)
    X_test_meta  = build_meta_features(X_test_cores, curr_base_models)
    
    # 3. Huấn luyện mô hình Meta cùng thuật toán
    meta_model = get_model_instance(name)
    if name == "Random Forest":
        meta_model.set_params(n_estimators=300)
    meta_model.fit(X_train_meta, y_train)
    
    # 4. Dự đoán trên tập Val và Test thông qua mô hình Meta
    y_val_pred  = meta_model.predict(X_val_meta)
    y_test_pred = meta_model.predict(X_test_meta)
    
    # 5. Tính toán metrics của mô hình tổng
    val_acc     = accuracy_score(y_val, y_val_pred)
    test_acc    = accuracy_score(y_test, y_test_pred)
    weighted_f1 = f1_score(y_test, y_test_pred, average="weighted", zero_division=0)
    macro_f1    = f1_score(y_test, y_test_pred, average="macro", zero_division=0)
    
    meta_results.append({
        "pipeline_name": name,
        "base_models": curr_base_models,
        "meta_model": meta_model,
        "predictions": y_test_pred,
        "val_accuracy": val_acc,
        "test_accuracy": test_acc,
        "weighted_f1": weighted_f1,
        "macro_f1": macro_f1
    })
    
    print(f"\n  [Meta Model: {name}] -> Val Acc: {val_acc:.4f} | Test Acc: {test_acc:.4f} | Macro F1: {macro_f1:.4f}")



  TRAINING & COMPARING COHESIVE PIPELINES

------------------------------------------------------------
  Training Stacked Pipeline: Random Forest
------------------------------------------------------------
  Base Models Accuracy on Test Set:
    ADLs & IADLs   : 0.5325
    Cognitive & Neurological Status: 0.5325
    Clinical Risk Assessments: 0.5325
    Mood & Behavioral Health: 0.5325
    Financial & Legal: 0.5325

  [Meta Model: Random Forest] -> Val Acc: 0.5324 | Test Acc: 0.5325 | Macro F1: 0.2317

------------------------------------------------------------
  Training Stacked Pipeline: Gradient Boosting
------------------------------------------------------------
  Base Models Accuracy on Test Set:
    ADLs & IADLs   : 0.5325
    Cognitive & Neurological Status: 0.5325
    Clinical Risk Assessments: 0.5325
    Mood & Behavioral Health: 0.5325
    Financial & Legal: 0.5325

  [Meta Model: Gradient Boosting] -> Val Acc: 0.5324 | Test Acc: 0.5325 | Macro F1: 0.2317

--------------

In [9]:
print("\n" + "=" * 75)
print("  PIPELINES COMPARISON SUMMARY")
print("=" * 75)

comparison_df = pd.DataFrame(meta_results)[[
    "pipeline_name", "val_accuracy", "test_accuracy", "weighted_f1", "macro_f1"
]].sort_values("macro_f1", ascending=False)

print(comparison_df.to_string(index=False))

# Lựa chọn Pipeline tốt nhất dựa trên Macro F1
best_idx = next(i for i, r in enumerate(meta_results) if r["pipeline_name"] == comparison_df.iloc[0]["pipeline_name"])
best_pipeline = meta_results[best_idx]
best_name = best_pipeline["pipeline_name"]

all_preds = best_pipeline["predictions"]
all_targets = y_test

print(f"\n Best Pipeline: {best_name}")
print(f"   Validation Accuracy: {best_pipeline['val_accuracy']:.4f}")
print(f"   Test Accuracy      : {best_pipeline['test_accuracy']:.4f}")
print(f"   Weighted F1-Score  : {best_pipeline['weighted_f1']:.4f}")
print(f"   Macro F1-Score     : {best_pipeline['macro_f1']:.4f}")

print(f"\nDetailed Classification Report — {best_name}:")
print(classification_report(all_targets, all_preds, target_names=[str(c) for c in le_target.classes_]))


print("\nConfusion Matrix:")
print(confusion_matrix(all_targets, all_preds))



  PIPELINES COMPARISON SUMMARY
    pipeline_name  val_accuracy  test_accuracy  weighted_f1  macro_f1
    Random Forest      0.532355       0.532533     0.370094  0.231657
Gradient Boosting      0.532355       0.532533     0.370094  0.231657
          XGBoost      0.532355       0.532533     0.370094  0.231657
         LightGBM      0.532355       0.532533     0.370094  0.231657

 Best Pipeline: Random Forest
   Validation Accuracy: 0.5324
   Test Accuracy      : 0.5325
   Weighted F1-Score  : 0.3701
   Macro F1-Score     : 0.2317

Detailed Classification Report — Random Forest:
              precision    recall  f1-score   support

         0.0       0.53      1.00      0.69      1064
         1.0       0.00      0.00      0.00       675
         2.0       0.00      0.00      0.00       259

    accuracy                           0.53      1998
   macro avg       0.18      0.33      0.23      1998
weighted avg       0.28      0.53      0.37      1998


Confusion Matrix:
[[1064    0   

In [10]:
print("\n" + "=" * 60)
print("  SAVING ALL TRAINED PIPELINES")
print("=" * 60)

import os

for r in meta_results:
    model_name = r["pipeline_name"]
    file_name = f"{model_name.lower().replace(' ', '_')}_pipeline.pkl"
    file_path = os.path.join(MODEL_SAVE_DIR, file_name)
    
    pipeline_data = {
        "pipeline_name": model_name,
        "base_models": r["base_models"],
        "meta_model": r["meta_model"],
        "feature_encoders": feature_encoders,
        "core_scalers": core_scalers,
        "core_cols_to_scale": core_cols_to_scale,
        "target_encoder": le_target,
        "core_features": CORE_FEATURES
    }
    
    joblib.dump(pipeline_data, file_path)
    print(f"Saved pipeline '{model_name}' to '{file_path}'")



  SAVING ALL TRAINED PIPELINES
Saved pipeline 'Random Forest' to 'D:\2026\MockProject_062026_NhomAI\model\random_forest_pipeline.pkl'
Saved pipeline 'Gradient Boosting' to 'D:\2026\MockProject_062026_NhomAI\model\gradient_boosting_pipeline.pkl'
Saved pipeline 'XGBoost' to 'D:\2026\MockProject_062026_NhomAI\model\xgboost_pipeline.pkl'
Saved pipeline 'LightGBM' to 'D:\2026\MockProject_062026_NhomAI\model\lightgbm_pipeline.pkl'
